# LAB08 · LAB09 · LAB10 — KPIs, auditoría y el pipeline del proyecto
## Bloque 2 · el día que se cierra el bloque

**Curso Big Data e IA Aplicada · Formación San Miguel**

---

Tres cosas hoy, y las tres van directas al proyecto:

| Parte | Qué produce | Para qué |
|---|---|---|
| **LAB08** | `salida/kpi_canal.csv` + **el párrafo de auditoría** | El indicador ① y la **Fase 3** |
| **LAB09** | El ancla en **el tercer motor** | La confianza en el dato |
| **LAB10** | `salida/ventas_limpio.parquet` | **Es la Fase 1. Sin esto, la Fase 2 no puntúa** |

---

### ✍️ LA PREDICCIÓN DEL DÍA — antes de tocar nada

Le vas a pedir a un asistente la facturación por ciudad sobre `ventas.csv`. **Su consulta será
sintácticamente impecable.**

**¿Cuánto crees que se va a desviar su Zaragoza de la tuya? Escribe una cifra en euros.**


---

## Paso 1 · El ancla · `BASE`



Como siempre, la celda 1. **Misma vista, letra por letra.**

In [ ]:
import duckdb



# ══════════════════════════════════════════════════════════════════════════

#  LIMPIO-v1 · el ancla del curso

#

#  Cada cuaderno del bloque 2 reconstruye la vista. Es idéntica LETRA A LETRA

#  a la del LAB06: esa igualdad ES el seguimiento del ancla.

#    (1) fuera precio_unitario <= 0        -> las 465 imposibles

#    (2) ciudad normalizada: TRIM + inicial en mayúscula

#    (3) ciudad vacía -> NULL, pero LA FILA SE CONSERVA

# ══════════════════════════════════════════════════════════════════════════



duckdb.sql("""CREATE OR REPLACE VIEW ventas_limpio AS

SELECT id_venta, fecha, id_cliente, id_producto, categoria, unidades, precio_unitario,

       CASE WHEN COALESCE(TRIM(ciudad), '') = '' THEN NULL

            ELSE UPPER(SUBSTR(TRIM(ciudad),1,1)) || LOWER(SUBSTR(TRIM(ciudad),2)) END AS ciudad,

       canal

FROM '../datasets/ventas.csv'

WHERE precio_unitario > 0""")



duckdb.sql("SELECT COUNT(*) AS filas_limpias FROM ventas_limpio").show()



# Esperado: 999535.  Si sale otra cosa, PARA: el resto del cuaderno no vale.

---



## Paso 2 · El cuadro de mando, en UNA consulta · `BASE`



Cuatro indicadores de una pasada.

In [ ]:
duckdb.sql("""SELECT COUNT(*)                                             AS ventas,

       ROUND(SUM(unidades*precio_unitario)/1e6, 1)          AS facturacion_M,

       ROUND(SUM(unidades*precio_unitario)/COUNT(*), 2)     AS ticket,

       COUNT(DISTINCT id_cliente)                           AS clientes_activos

FROM ventas_limpio""").show()



# Esperado: 999535 · 429.9 · 430.09 · 99996

> 📌 Mira la última columna: **99.996 clientes activos** de 100.000. Son los mismos cuatro
> fantasmas del LAB07, apareciendo ahora **dentro de un KPI**. Un indicador bien construido arrastra
> las historias que ya conoces.



---



## Paso 3 · KPIs por dimensión · `BASE`



Y fíjate en el `IS NOT NULL` de la segunda consulta: **el silencio de los NULL, convertido en
decisión declarada.**

In [ ]:
duckdb.sql("""SELECT canal, ROUND(SUM(unidades*precio_unitario)/1e6, 1) AS M

FROM ventas_limpio

GROUP BY canal ORDER BY M DESC""").show()



# Esperado: tienda 214.3 · web 143.3 · movil 72.3



duckdb.sql("""SELECT ciudad, ROUND(SUM(unidades*precio_unitario), 2) AS facturacion

FROM ventas_limpio

WHERE ciudad IS NOT NULL

GROUP BY ciudad ORDER BY facturacion DESC

LIMIT 4""").show()



# Esperado: Zaragoza 146920182.71 · Madrid 59873119.57 · Barcelona 51325765.64 · Huesca 42491623.30

### ✍️ ANOTA TU ZARAGOZA



**146.920.182,71 €**



Escríbelo. Dentro de diez minutos vas a compararlo contra lo que te diga una IA, y **ese contraste
es el ejercicio**.





> 💡 Las 3.030 ventas con ciudad `NULL` **desaparecen en silencio** de este análisis. Ese silencio
> tiene que ser **una decisión declarada** en tu informe —«análisis sobre las 996.505 ventas con
> ciudad conocida»— y no un accidente.
>
> **La frase de oficio:** cuando un número baile entre dos consultas «equivalentes», busca primero
> un `NULL` y después un JOIN multiplicador.



---



## Paso 4 · Publicar · `BASE`



`COPY (consulta) TO 'ruta' (HEADER)` convierte una consulta en **un fichero con ruta**: la salida
deja de ser una pantalla y pasa a ser un **entregable consumible por otros**.



> 🎯 Tu flujo de n8n del Bloque 3 leerá **exactamente este fichero**. Y `datasets/salida/` es, desde
> hoy, la carpeta oficial de resultados del proyecto.

In [ ]:
import os
os.makedirs("../datasets/salida", exist_ok=True)

duckdb.sql("""COPY (
    SELECT canal,
           COUNT(*)                                          AS ventas,
           ROUND(SUM(unidades*precio_unitario)/1e6, 1)       AS millones,
           ROUND(SUM(unidades*precio_unitario)/COUNT(*), 2)  AS ticket
    FROM ventas_limpio
    GROUP BY canal ORDER BY millones DESC
) TO '../datasets/salida/kpi_canal.csv' (HEADER)""")

print("Publicado: ../datasets/salida/kpi_canal.csv")
print()
print(open("../datasets/salida/kpi_canal.csv", encoding="utf-8").read())

# Esperado: tienda 214.3 · web 143.3 · movil 72.3
# Esos tres números son los que el informe automático tendrá que citar sin inventar.

> 💡 Compruébalo si quieres desde la ▸ **Terminal** con un `cat`. **Tus dos mundos, colaborando:**
> el fichero que ha escrito SQL lo lee la terminal del jueves pasado sin enterarse de nada.



---

---



# ⚠️ Paso 5 · LA AUDITORÍA · `COMPLETA`

## Esta es la pieza que se puntúa por escrito en la Fase 3 del proyecto



**El protocolo son cuatro pasos. Hazlos en orden y sin adelantarte.**



---



### ① Pídeselo, **sin darle el ancla**



En tu asistente (plantilla exacta en `plantillas/prompt_auditoria_sql.txt`):



> *«Escríbeme la consulta SQL de la facturación total por ciudad sobre `ventas.csv`, ordenada de
> mayor a menor.»*



### ② Pega su consulta en la celda de abajo y **ejecútala TAL CUAL**



**Sin retocarla. Sin arreglarle nada.** Si la corriges antes de ejecutarla, has destruido la prueba.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════

#  PEGA AQUÍ LA CONSULTA DE TU IA, TAL CUAL TE LA HAYA DADO.

#  No la retoques. No le arregles la ruta si se equivoca: anótalo y arréglala

#  DESPUÉS, dejando constancia de que se equivocó.

# ══════════════════════════════════════════════════════════════════════════



duckdb.sql("""



""").show()

### ③ Compara contra tu número canónico



| | Zaragoza |
|---|---|
| **La tuya** (sobre LIMPIO-v1) | **146.920.182,71 €** |
| **La suya** | ✍️ |
| **La diferencia** | ✍️ |



**Qué esperar — la trampa anunciada:** su Zaragoza dirá **≈145,65 M**. Es **el número sucio**, el
que tú mismo calculaste con `awk` en el LAB03.



**La IA no conoce LIMPIO-v1 salvo que se lo pegues.** No filtró los 465 imposibles ni devolvió a
casa a las zaragozas descarriadas.



✍️ **Lee su SQL y encuentra LA decisión que tomó por ti. ¿Filtró la suciedad? ¿Normalizó la ciudad?

¿Qué hizo con los NULL?**





---



### ④ Repite la petición **CON el ancla pegada**, y verifica la convergencia



> *«Sobre `ventas.csv`, usando esta definición de limpio: (1) fuera `precio_unitario <= 0`;
> (2) ciudad con `TRIM` y primera letra en mayúscula; (3) ciudad vacía a `NULL` conservando la fila.
>
> Dame la facturación por ciudad ordenada de mayor a menor.»*



**Debe converger a 146,92 M.**



---



### ✍️ EL PÁRRAFO DE AUDITORÍA — esto es literalmente lo que se puntúa



**Escribe tres cosas: qué asumió, qué cambió al darle el ancla, y cuánto valía la diferencia.**





> 💡 **1,27 M€ de diferencia por una definición sin anclar.** No por un error de sintaxis: su código
> era **impecable** y se ejecutó **sin un solo aviso**.
>
> **La corrección gramatical no garantiza la corrección de negocio.** Es la tercera vez que lo ves
> en este curso —la primera fue el stock de la sesión 3, la segunda el tamaño del Parquet— y las
> tres veces lo has cazado **del mismo modo: contando contra un número que ya conocías.**
>
> ⚠️ **Si tu IA acierta a la primera** (alguna limpia por iniciativa): **también es hallazgo.**
>
> Pregúntale **qué asumió** y compáralo con el ancla cláusula a cláusula: ¿coincide su limpieza con
> la tuya, o acertó el total **por otra vía**? Escríbelo igual: has auditado igual de bien.



---



## Paso 6 · Mes pico por segmento · `RETO` · a casa



¿Coinciden los tres segmentos en su mejor mes? (JOIN + `STRFTIME` + `GROUP BY` doble — **todas las
piezas ya son tuyas**.)



---

---



# 🏁 Qué te llevas



- Un **cuadro de KPIs** reproducible y **publicado con ruta**.
- La lógica del `NULL` convertida en **decisión declarada**, no en accidente.
- Y el **protocolo de auditoría** completo: su SQL tal cual → contra tu ancla → busca su supuesto →

  repite con el ancla.



> **La frase de la sesión:** *no auditas ortografía — auditas supuestos. Esa es la competencia.*

---
---

# LAB09 + LAB10 · Spark y el pipeline ETL

**Las dos ideas de diseño, en dos frases.**

**Inmutabilidad.** Un DataFrame **nunca se modifica**: cada operación devuelve uno nuevo. Es **la
primera ley del curso** elevada a arquitectura — y no por estética: si una máquina muere a mitad de
trabajo, Spark recalcula su trozo desde el original **porque nada se sobrescribió**.

**Pereza.** Las transformaciones **no calculan nada**: construyen un plan. Solo una **acción**
(`count`, `show`, `write`) lo dispara. No vas al súper cada vez que apuntas un ingrediente.

Y la palabra cara: **el shuffle**. Un `groupBy` exige que todas las filas de «web» acaben juntas,
estén en la partición que estén. Es tu `sort` externo **con la red de por medio**. En el plan se
llama `Exchange`.

### ✍️ Dos predicciones

**① ¿El `count()` de Spark dará lo mismo que el `COUNT(*)` de DuckDB?**


**② En la MISMA agregación, ¿quién tarda más, Spark en local o DuckDB? ¿Por qué?**


> ⚠️ **El primer arranque de Spark tarda 30–60 segundos.** El semáforo `[*]` significa
> **trabajando**, no colgado.

---

---



# LAB09 · Primeros DataFrames



## Paso 1 · La sesión · `BASE`



`local[*]` es tu cuadrilla: **los núcleos de tu máquina**. Pequeña, pero cuadrilla de verdad —
misma arquitectura, mismo plan, **mismo código** que en un clúster de mil máquinas.



> La `config` de memoria salva a los puestos justos. **Déjala aunque el tuyo vaya sobrado.**

In [ ]:
from pyspark.sql import SparkSession



spark = (SparkSession.builder.appName('curso')

         .master('local[*]')                              # la cuadrilla: TUS núcleos

         .config('spark.driver.memory', '1g')             # el techo del jefe de obra

         .config('spark.ui.showConsoleProgress', 'false')

         .getOrCreate())



print("Spark", spark.version)

---



## Paso 2 · Leer el CSV · `BASE`



`header` dice que la primera línea son **nombres**; `inferSchema` pide **adivinar los tipos**
escaneando el fichero — comodidad con coste. En producción el esquema se declara.

In [ ]:
df = (spark.read.option('header', True).option('inferSchema', True)

      .csv('../datasets/ventas.csv'))



df.printSchema()

print("filas:", df.count())



# Esperado: 1000000  -> igual que SQL: la cabecera son nombres (predicción ①)

---



## Paso 3 · ⚓ LIMPIO-v1, dialecto Spark · `BASE`



**La MISMA ancla, tercera sintaxis.** Compárala cláusula a cláusula con la vista del LAB06:



| LIMPIO-v1 | SQL (LAB06) | Spark (aquí) |
|---|---|---|
| Fuera los precios imposibles | `WHERE` | `.filter(...)` |
| El «si» de la normalización | `CASE WHEN` | `F.when(...).otherwise(...)` |
| Inicial en mayúscula | `UPPER(SUBSTR(...))` | `F.initcap(F.trim(...))` |

In [ ]:
from pyspark.sql import functions as F



limpio = (df.filter(F.col('precio_unitario') > 0)

            .withColumn('ciudad',

                        F.when(F.trim(F.coalesce(F.col('ciudad'), F.lit(''))) == '', None)

                         .otherwise(F.initcap(F.trim('ciudad')))))



print("filas limpias:", limpio.count())



# Esperado: 999535  -> TERCERA herramienta, MISMO número.

### ✅ 999.535, por tercera vez



`awk` en la terminal. `SELECT` en DuckDB. `filter` en Spark. **Tres motores que no se conocen entre
sí, un solo número.**



✍️ **¿Te ha salido? Escríbelo, y escribe al lado con qué tres herramientas lo has obtenido ya.**





> 🎯 **Así se confía en un dato.** No porque el código sea bonito: porque **dos caminos
> independientes llegan al mismo sitio**. Es lo que te van a puntuar en la Fase 1 del proyecto, y es
> lo que hace un ingeniero de datos cuando nadie le regala la confianza.



---



## Paso 4 · La agregación de siempre · `BASE`

In [ ]:
(limpio.groupBy('canal')

       .agg(F.round(F.sum(F.col('unidades') * F.col('precio_unitario')) / 1e6, 1).alias('millones'))

       .orderBy(F.desc('millones'))

       .show())



# Esperado: tienda 214.3 · web 143.3 · movil 72.3   -> clavado a verificacion_b2.sh

---



## Paso 5 · La pereza, demostrada · `COMPLETA` · *(predicción ②)*



Un plan que **no ha calculado nada**. Busca la palabra `Exchange` en la salida: **ese es el
shuffle**.

In [ ]:
plan = (limpio.groupBy('categoria')

              .agg(F.sum(F.col('unidades') * F.col('precio_unitario')).alias('fact')))



print(type(plan))     # DataFrame -- y NO ha leído ni una fila: es un PLAN

plan.explain()        # el plan físico. Busca: Exchange

✍️ **¿Encontraste el `Exchange`? ¿Antes o después del agregado?**





---



## Paso 6 · La acción dispara · `COMPLETA`

In [ ]:
plan.orderBy(F.desc('fact')).show(5)



# Ahora SÍ trabaja. Transformaciones que apuntan, acciones que van al súper.

### 💡 La frontera, medida por ti



Cronométralo mentalmente contra tu DuckDB de ayer.



✍️ **¿Quién ha ganado? ¿Coincide con tu predicción ③?**





> 🎯 **En local y con 1 GB, DuckDB gana casi seguro.** Y esa lentitud relativa **es un DATO, no una
> decepción**: Spark amortiza su maquinaria cuando los datos son MUCHO mayores.
>
> **La regla de bolsillo, para decirla sin dudar:** si cabe en tu disco, **DuckDB**; si necesitas un
> clúster para **almacenarlo**, Spark.
>
> 💡 Y en producción **la única línea que cambia** es el `master(…)`: de `local[*]` a la dirección
> de un coordinador de clúster. **Tu código DataFrame no se entera.** Esa portabilidad ES el
> producto, y es la razón de que aprender en local sea aprender de verdad.



---

---



# LAB10 · El pipeline ETL del proyecto



**Extract** (leer las fuentes tal cual llegan) → **Transform** (aplicar las reglas de negocio — y
**la T de tu pipeline es, literalmente, LIMPIO-v1**) → **Load** (publicar en el formato del
presente: Parquet).



Es **EL** patrón de la ingeniería de datos. Millones de pipelines nocturnos en el mundo son
exactamente esto con más ceros.



## Pasos 1 y 2 · E y T · `BASE`



**Ya están hechos**: son `df` (1.000.000) y `limpio` (999.535), las mismas celdas de arriba — ahora
con nombre de fase de un pipeline.



## Paso 3 · L: el Parquet maestro · `BASE`

In [ ]:
(limpio.write.mode('overwrite')

       .parquet('../datasets/salida/ventas_limpio.parquet'))



print("Publicado el almacén limpio.")



# 'overwrite' = re-ejecutable sin miedo: el pipeline puede correr mil veces.

# Eso se llama REPRODUCIBILIDAD, y es lo que se puntúa en la Fase 1.

> ⚠️ **Sorpresa de mundo real:** Spark **no** escribe un fichero, escribe **una carpeta** con varios
> `part-*.parquet` — un fichero por partición de trabajo, porque también paraleliza al escribir. El
> fichero único del LAB05 era el caso de juguete.
>
> Se lee entera con el comodín `'carpeta/*.parquet'`. Míralo desde la ▸ **Terminal**:
> `ls ../datasets/salida/ventas_limpio.parquet/` — verás los `part-*` y un `_SUCCESS`.



---



## Paso 4 · Las tablas de KPIs · `BASE`

In [ ]:
(limpio.groupBy('categoria')

       .agg(F.round(F.sum(F.col('unidades') * F.col('precio_unitario')) / 1e6, 2).alias('millones'))

       .orderBy(F.desc('millones'))

       .write.mode('overwrite')

       .parquet('../datasets/salida/kpi_categoria.parquet'))



print("Publicado el cuadro de categorías.")

---



## ⭐ Paso 5 · LA VERIFICACIÓN CRUZADA FINAL · `COMPLETA`



**El otro motor audita lo que acaba de publicar el primero.** Dos motores que no se conocen, un
formato abierto entre medias.

In [ ]:
import duckdb



duckdb.sql("SELECT COUNT(*) AS filas FROM '../datasets/salida/ventas_limpio.parquet/*.parquet'").show()

# Esperado: 999535



duckdb.sql("""SELECT categoria, millones

FROM '../datasets/salida/kpi_categoria.parquet/*.parquet'

ORDER BY millones DESC""").show()

# Esperado: informatica 256.5 · hogar 61.8 · jardin 58.79 · deporte 43.55 · papeleria 9.26

### 💡 Interoperabilidad



**Esa palabra es la razón por la que Parquet ganó** — y la acabas de demostrar con tu propio
pipeline, no leído en ningún sitio.



✍️ **Escribe la cadena completa: qué herramienta produjo el dato, cuál lo leyó, y qué número
compartieron.**





---

In [ ]:
spark.stop()

print("Spark apagado. Memoria devuelta.")



# Ley en máquinas compartidas: lo que enciendes, lo apagas.

---



## Paso 6 · El particionado · `RETO` · a casa



Añade `.partitionBy('canal')` antes del `.parquet(…)`, republica, y compara el `ls` de ambas
versiones. **Acabas de organizar tu primer lago de datos.**



✍️ **Pregunta de cierre: ¿por qué un filtro `canal='web'` sería ahora más rápido?**





---

---





# 🔍 CONSULTA · Bloque A



**Cuatro preguntas, una de cada etiqueta.**



---



**A1 · ⚙️ MÁQUINA**



✍️ **Tu Zaragoza, la de la IA, y la diferencia en euros. Y las filas del Parquet maestro releído
por DuckDB.**





---



**A2 · 📝 CRITERIO**



✍️ **«Su SQL era correcto y su número estaba mal.» Explica esa frase en dos líneas, a alguien que
no sabe SQL.**





---



**A3 · 🗂️ RAG**



> *Según el manual, ¿qué distingue a un KPI de «un número interesante»? **Cítame el apartado.***



✍️





---



**A4 · 🤖 ASISTENTE**



> *¿Por qué un `groupBy` de Spark necesita un shuffle y un `filter` no?*



**Y ahora audítalo:** compara su explicación con el `Exchange` que has visto en tu propio
`explain()`. ¿Dice lo mismo que tu plan?



✍️





---

---



# 📦 Entregable · CIERRE DEL BLOQUE 2



**`Ctrl+S` en los dos cuadernos de hoy antes de archivar.**



### Lo que tiene que estar hecho



| # | Contenido | ¿Hecho? |
|---|---|---|
| 1 | **El ancla por tres vías**: 999.535 en awk, en SQL y en Spark | |
| 2 | **La auditoría del LAB08**: tu Zaragoza, la suya y el párrafo | |
| 3 | **`salida/ventas_limpio.parquet`** publicado y releído por DuckDB | |
| 4 | **`kpi_canal.csv`** publicado | |
| 5 | Los bloques **S, N, A y E** contestados | |

---

---



# 🏁 Cierre del Bloque 2 · haz inventario



Entraste hablando terminal y sales hablando **el idioma de la industria**:



- **SQL**: las seis cláusulas con su orden real, `FILTER`, fechas, `CASE`, vistas, tipos y el baile

  del decimal explicado.

- **JOINs**: claves, cardinalidades, `INNER` y `LEFT`, el anti-join, el control del `COUNT` y la

  lógica de tres valores del `NULL`.

- **Calidad de datos**: una definición **anclada**, con consecuencias medidas y verificación

  ejecutable.

- **Spark**: la historia de MapReduce a Spark, driver y executors, la pereza, el shuffle, escribir

  Parquet — y **la frontera honesta con DuckDB**.

- **IA aplicada**: el protocolo de auditoría de SQL generado, **con ancla y en euros**.



### En el idioma en que se escriben los currículums



> Modelado y consulta analítica con SQL sobre datasets de millones de registros · definición y
> gobierno de reglas de calidad de datos (data quality anchoring) · construcción de pipelines ETL
> reproducibles con Apache Spark y publicación en formato columnar · verificación cruzada de
> resultados entre motores heterogéneos · auditoría de código SQL generado por IA.
>
> **La frase de salida del bloque — dila en voz alta:**
>
> *«Puedo conectar tablas, definir limpio por escrito, publicar KPIs y reproducir el resultado en
> dos motores. Me falta profundidad, no base.»*



**El Bloque 3 empieza exactamente donde esto termina:** la IA **dentro** del pipeline que acabas de
construir.

In [ ]:
import shutil, os, glob, json

SESION = 8                      # numeracion del MANUAL (sesiones 4, 5 y 6)

# ══════════════════════════════════════════════════════════════════════════
#  GUARDIÁN · ¿está en el disco lo que ves en pantalla?
#
#  Esta celda copia el FICHERO DEL DISCO, no lo que tienes delante. Jupyter
#  guarda solo cada pocos minutos: si archivas antes de un Ctrl+S, entregas
#  el cuaderno SIN tus resultados y el HTML sale vacío.
# ══════════════════════════════════════════════════════════════════════════

def resultados_en_disco(ruta):
    try:
        nb = json.load(open(ruta, encoding="utf-8"))
    except Exception:
        return 0, 0
    codigo = [c for c in nb["cells"] if c["cell_type"] == "code"]
    return sum(1 for c in codigo if c.get("outputs")), len(codigo)

cuadernos = [f for f in glob.glob("*lab08*.ipynb") if ".ipynb_checkpoints" not in f]
listo = bool(cuadernos)

if not cuadernos:
    print("  No encuentro los cuadernos de esta sesión en esta carpeta.")

for cuaderno in cuadernos:
    hechas, total = resultados_en_disco(cuaderno)
    print(f"  {cuaderno}: {hechas} de {total} celdas con resultados en el disco")
    if hechas == 0:
        listo = False

if not listo:
    print()
    print("  " + "=" * 68)
    print("   PARA AQUI. No he archivado nada.")
    print("   Pulsa  Ctrl+S  y vuelve a ejecutar ESTA celda.")
    print("  " + "=" * 68)
else:
    os.makedirs("entregables", exist_ok=True)
    piezas = cuadernos + [f for f in ["mi_bitacora.ipynb"] if os.path.exists(f)]
    for pieza in piezas:
        destino = f"entregables/S{SESION:02d}_{os.path.basename(pieza)}"
        shutil.copy(pieza, destino)
        print("  copiado:", destino)

In [ ]:
import glob, os

# Exporta a HTML. El HTML conserva las salidas incrustadas: se manda por
# correo y se ve entero, sin entorno, sin kernel, sin nada.
for cuaderno in glob.glob("*lab08*.ipynb"):
    if ".ipynb_checkpoints" in cuaderno:
        continue
    salida = f"S{SESION:02d}_" + os.path.splitext(os.path.basename(cuaderno))[0]
    !jupyter nbconvert --to html --output-dir entregables --output {salida} "{cuaderno}"

In [ ]:
import glob, os, re

# ══════════════════════════════════════════════════════════════════════════
#  LA COMPROBACIÓN QUE CIERRA EL CÍRCULO
#
#  Un cuaderno ejecutado deja en el HTML el número de cada celda: [1]:, [2]:...
#  Si no hay ninguno, el HTML NO lleva tus resultados.
#  El TAMAÑO del fichero engaña; este número, no.
# ══════════════════════════════════════════════════════════════════════════

def celdas_ejecutadas(ruta_html):
    h = open(ruta_html, encoding="utf-8", errors="ignore").read()
    return len(re.findall(r"\[[0-9]+\]:", h)), h.count("data:image/png;base64")

htmls = sorted(glob.glob("entregables/*.html"))
vacios = []

if not htmls:
    print("  No hay ningún HTML. ¿Ejecutaste la celda anterior?")

for h in htmls:
    ejecutadas, graficas = celdas_ejecutadas(h)
    kb = os.path.getsize(h) / 1024
    if ejecutadas == 0:
        vacios.append(os.path.basename(h))
    estado = "OK    " if ejecutadas else "VACIO "
    print(f"  {estado} {os.path.basename(h):<34} {ejecutadas:>3} celdas ejecutadas · {kb:.0f} KB")

print()
if vacios:
    print("  Estos HTML no llevan resultados:", ", ".join(vacios))
    print("  Ctrl+S y repite las dos celdas anteriores.")
elif htmls:
    print("  Todo correcto. Ya puedes empaquetar.")

In [ ]:
import os, re, glob, zipfile

APELLIDO_NOMBRE = "PEREZ_Ana"        # <-- pon el tuyo ANTES de ejecutar

# Última verificación antes de empaquetar: ningún HTML puede estar vacío.
vacios = [os.path.basename(h) for h in glob.glob("entregables/*.html")
          if not re.findall(r"\[[0-9]+\]:", open(h, encoding="utf-8", errors="ignore").read())]

if vacios:
    print("  " + "=" * 68)
    print("   NO EMPAQUETO: estos HTML no llevan resultados.")
    for v in vacios:
        print("     -", v)
    print("   Pulsa Ctrl+S y repite las tres celdas anteriores.")
    print("  " + "=" * 68)
else:
    nombre_zip = f"{APELLIDO_NOMBRE}_bloque2.zip"

    # A mano en vez de con make_archive, para poder EXCLUIR los checkpoints
    # de Jupyter: si no, se cuelan y multiplican el tamaño de la entrega.
    with zipfile.ZipFile(nombre_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for raiz, carpetas, ficheros in os.walk("entregables"):
            carpetas[:] = [c for c in carpetas if c != ".ipynb_checkpoints"]
            for f in ficheros:
                ruta = os.path.join(raiz, f)
                z.write(ruta, os.path.relpath(ruta, "entregables"))

    print(f"Creado: {nombre_zip}  ({os.path.getsize(nombre_zip)/1024:.0f} KB)")
    print()
    print("Contenido:")
    with zipfile.ZipFile(nombre_zip) as z:
        for n in sorted(z.namelist()):
            print("   ", n)
    print()
    print("Clic derecho sobre el .zip -> Download -> y a Moodle.")